# 03 — Modeling, Validation Tuning, Test Evaluation & Explainability

**Project:** Cinema Ticket Demand Forecasting (`T-48h`)  
**Workflow:** Summarizes validation hyperparameter search (`reports/experiments.csv`), evaluates all models once on the untouched test set (`reports/metrics.json`), inspects cold-start/capacity/error slices, and reviews global + local explanations (`reports/explanations.json`).

In [1]:
import json
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from src.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
exp_df = pd.read_csv(PROJECT_ROOT / cfg["paths"]["experiment_log"])
with (PROJECT_ROOT / cfg["paths"]["metrics_report"]).open("r", encoding="utf-8") as fh:
    metrics = json.load(fh)
with (PROJECT_ROOT / "reports" / "explanations.json").open("r", encoding="utf-8") as fh:
    explanations = json.load(fh)
with (PROJECT_ROOT / cfg["paths"]["model_metadata"]).open("r", encoding="utf-8") as fh:
    metadata = json.load(fh)

print("Selected Model:", metadata["best_model_name"], metadata["best_params"])
print("CQR 80% Delta (occ):", metadata["cqr_delta_occ"])

Selected Model: HistGradientBoosting {'max_iter': 250, 'max_depth': 4, 'learning_rate': 0.08, 'min_samples_leaf': 40, 'random_state': 42}
CQR 80% Delta (occ): 0.01512


## 1. Validation Experiment Log (Sorted by Validation MAE)

In [2]:
cols = ["model_family", "params", "cv_mae_tickets", "val_mae_tickets", "val_rmse_tickets", "val_wape_pct", "val_mae_occ_pts"]
print(exp_df[cols].sort_values("val_mae_tickets").to_string(index=False))

        model_family                                                                                               params  cv_mae_tickets  val_mae_tickets  val_rmse_tickets  val_wape_pct  val_mae_occ_pts
HistGradientBoosting {"max_iter": 250, "max_depth": 4, "learning_rate": 0.08, "min_samples_leaf": 40, "random_state": 42}         10.3212          10.9181           15.2675       20.0882           6.4167
HistGradientBoosting {"max_iter": 250, "max_depth": 4, "learning_rate": 0.08, "min_samples_leaf": 20, "random_state": 42}             NaN          10.9283           15.3872       20.1071           6.4183
HistGradientBoosting {"max_iter": 250, "max_depth": 4, "learning_rate": 0.05, "min_samples_leaf": 20, "random_state": 42}             NaN          10.9326           15.3571       20.1150           6.4121
HistGradientBoosting {"max_iter": 250, "max_depth": 4, "learning_rate": 0.05, "min_samples_leaf": 40, "random_state": 42}             NaN          10.9614           15.3363       20.16

## 2. Untouched Test Set Comparison

In [3]:
test_comp = pd.DataFrame(metrics["model_comparison_test"])
display_cols = [
    "model", "mae_tickets", "rmse_tickets", "wape_pct", "mae_occ_pts",
    "within_10_occ_pts_pct", "clip_outside_pct", "sold_out_precision_point",
    "sold_out_recall_point", "mean_business_cost_per_show"
]
print(test_comp[display_cols].to_string(index=False))

               model  mae_tickets  rmse_tickets  wape_pct  mae_occ_pts  within_10_occ_pts_pct  clip_outside_pct  sold_out_precision_point  sold_out_recall_point  mean_business_cost_per_show
 Baseline (Hist Avg)      26.1756       37.1589   60.8551      15.1425                  39.27             0.000                    0.0000                 0.0000                       139.47
               Ridge      12.6574       17.9465   29.4270       7.3576                  73.41             6.122                    0.0000                 0.0000                        68.91
        RandomForest      10.0770       14.3586   23.4278       5.8553                  83.64             0.000                    0.2857                 0.3636                        59.88
HistGradientBoosting       9.9330       13.7868   23.0930       5.8211                  83.52             0.000                    0.3684                 0.3182                        59.60


## 3. Cold-Start, Capacity Bucket & Error Slices (Test Set)

In [4]:
print("=== Cold-Start / History Depth (Best Model) ===")
print(pd.DataFrame(metrics["best_model_slices_test"]["cold_start"]).T.to_string())
print("\n=== Capacity Buckets (Best Model) ===")
print(pd.DataFrame(metrics["best_model_slices_test"]["capacity_bucket"]).T.to_string())
print("\n=== Top 10 Worst Test Errors ===")
print(pd.DataFrame(metrics["worst_20_shows"]).head(10).to_string(index=False))

=== Cold-Start / History Depth (Best Model) ===
             n_shows  mae_tickets  rmse_tickets  wape_pct  mae_occ_pts  within_10_occ_pts_pct  clip_outside_pct  total_business_cost  mean_business_cost_per_show  sold_out_actual_count  sold_out_precision_point  sold_out_recall_point
new_movie      226.0      16.1083       21.3354   15.1617       9.1689                  58.85               0.0             22563.02                        99.84                   11.0                    0.3684                 0.6364
low_history      5.0      18.0493       21.2633   15.3742       8.7361                  60.00               0.0               274.85                        54.97                    0.0                    0.0000                 0.0000
mature        3003.0       9.4547       13.0264   24.7957       5.5643                  85.41               0.0            169903.06                        56.58                   11.0                    0.0000                 0.0000

=== Capacity Bu

## 4. Global & Local Explainability

In [5]:
print("=== Top 10 Permutation Importance Features ===")
print(pd.DataFrame(explanations["global_permutation_importance"]).head(10).to_string(index=False))
print("\n=== Local Accurate Case ===")
print(explanations["local_accurate_case"]["plain_language_explanation"])
print("\n=== Local Large Error Case ===")
print(explanations["local_large_error_case"]["plain_language_explanation"])

=== Top 10 Permutation Importance Features ===
                  feature  importance_occ_mae  std_occ_mae
           movie_age_days              6.6941       0.0703
   slot_weekday_occ_prior              1.8886       0.0952
                  weekday              0.8779       0.0470
      movie_slot_occ_mean              0.5418       0.0210
        movie_occ_mean_7d              0.4289       0.0303
       movie_occ_mean_14d              0.2496       0.0212
   movie_weekday_occ_mean              0.2155       0.0103
         is_special_event              0.1511       0.0276
movie_effective_occ_prior              0.1159       0.0159
                 capacity              0.0842       0.0086

=== Local Accurate Case ===
For Comedy Feature #35 (Comedy, age 3d) in SCR-01 (Standard, cap 85) during evening on 2025-04-04, the model predicted 60 tickets (80% interval: 51-76) and actual sales were 57 (error -2.9). Key drivers: movie_age_days (+0.216 occ), slot (+0.139 occ), movie_slot_occ_mean (+0